# 1. Dependencies & Configuration

In [3]:
import logging
import sys
import warnings
from prophet import Prophet
import numpy as np
import pandas as pd

# Suppress noisy logs from Prophet and Stan
logging.getLogger("prophet").setLevel(logging.ERROR)
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

# Global configurations
RAW_DATA_PATH = "walmart_final_train_202605212105"
OUTPUT_PATH = "walmart_sales_predictions_murni_future.csv"
FORECAST_HORIZON_WEEKS = 12
TOP_N_COMBINATIONS = 10

print("Data Successfully Loaded!")

Data Successfully Loaded!


# 2. Data Loading & Sanitization

In [5]:
print("\n Loading raw data and sanitising dates...")

RAW_DATA_PATH = "walmart_final_train_202605212105.csv"
OUTPUT_PATH   = "walmart_sales_predictions_murni_future.csv"
FORECAST_HORIZON_WEEKS = 12
TOP_N_COMBINATIONS     = 10

try:
    df_raw = pd.read_csv(RAW_DATA_PATH)
    print(f"   Raw file loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
except FileNotFoundError:
    sys.exit(f"  ERROR: '{RAW_DATA_PATH}' not found. "
             "Check the file path and re-run.")

# Handle common column name variants from different SQL export configurations.
column_aliases = {
    "Date"              : "Transaction_Date",
    "date"              : "Transaction_Date",
    "transaction_date"  : "Transaction_Date",
    "TRANSACTION_DATE"  : "Transaction_Date",
    "Sales"             : "Weekly_Sales",
    "weekly_sales"      : "Weekly_Sales",
    "WEEKLY_SALES"      : "Weekly_Sales",
}
df_raw.rename(columns=column_aliases, inplace=True)

# Validate required columns exist
required_cols = {"Transaction_Date", "Weekly_Sales", "Store", "Dept"}
missing_cols  = required_cols - set(df_raw.columns)
if missing_cols:
    sys.exit(f"  ✗ ERROR: Required columns missing from CSV: {missing_cols}\n"
             f"    Available columns: {list(df_raw.columns)}")

print("  → Parsing and sanitising 'Transaction_Date' column...")

df_raw["Transaction_Date"] = pd.to_datetime(
    df_raw["Transaction_Date"],
    dayfirst=False,
    infer_datetime_format=True,
    errors="coerce"
)

# Count and report any unparseable dates
nat_count = df_raw["Transaction_Date"].isna().sum()
if nat_count > 0:
    print(f"   WARNING: {nat_count} rows had unparseable dates → DROPPED.")
    df_raw.dropna(subset=["Transaction_Date"], inplace=True)

# .strftime('%Y-%m-%d') returns a plain Python str, not a timestamp.
df_raw["Transaction_Date"] = df_raw["Transaction_Date"].dt.strftime("%Y-%m-%d")

print(f"   Date sanitisation complete. "
      f"Sample: {df_raw['Transaction_Date'].iloc[0]!r}  ← must be 'YYYY-MM-DD'")

# Numeric type enforcement
df_raw["Weekly_Sales"] = pd.to_numeric(df_raw["Weekly_Sales"], errors="coerce").fillna(0.0)
df_raw["Store"]        = df_raw["Store"].astype(int)
df_raw["Dept"]         = df_raw["Dept"].astype(int)

# Handle optional regressor columns
OPTIONAL_REGRESSORS = ["CPI", "Unemployment", "Temperature", "Fuel_Price",
                        "IsHoliday", "MarkDown1", "MarkDown2",
                        "MarkDown3", "MarkDown4", "MarkDown5"]
available_regressors = [c for c in OPTIONAL_REGRESSORS if c in df_raw.columns]
print(f"   Available external regressors: "
      f"{available_regressors if available_regressors else 'None — base model only'}")

print(f"\n   Clean dataset ready: {df_raw.shape[0]:,} rows")
print(f"     Date range: {df_raw['Transaction_Date'].min()} → "
      f"{df_raw['Transaction_Date'].max()}")


 Loading raw data and sanitising dates...
   Raw file loaded: 421,570 rows × 16 columns
  → Parsing and sanitising 'Transaction_Date' column...
   Date sanitisation complete. Sample: '2010-02-05'  ← must be 'YYYY-MM-DD'
   Available external regressors: ['CPI', 'Unemployment', 'Temperature', 'Fuel_Price', 'IsHoliday', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']

   Clean dataset ready: 421,570 rows
     Date range: 2010-02-05 → 2012-10-26


# 3. Identify Top N High-Volume Combinations

In [6]:
# Aggregate global sales to identify top under-scoped combinations dynamically
combo_sales = (
    df_raw.groupby(["Store", "Dept"], as_index=False)["Weekly_Sales"]
    .sum()
    .rename(columns={"Weekly_Sales": "Total_Sales"})
    .sort_values("Total_Sales", ascending=False)
    .reset_index(drop=True)
)

top10_combos = combo_sales.head(TOP_N_COMBINATIONS).copy()

# Map configurations for verification logs
print(f"\n{'Rank':<6} {'Store':<8} {'Dept':<8} {'Total_Sales':>15}")
print(f"{'-'*43}")
for rank, row in top10_combos.iterrows():
    print(f"{rank+1:<6} {int(row['Store']):<8} {int(row['Dept']):<8} ${row['Total_Sales']:>14,.2f}")

# Hash map set deployment for O(1) loop membership validation
top10_set = set(zip(top10_combos["Store"].astype(int), top10_combos["Dept"].astype(int)))


Rank   Store    Dept         Total_Sales
-------------------------------------------
1      14       92       $ 26,101,497.57
2      2        92       $ 23,572,153.06
3      20       92       $ 23,542,624.99
4      13       92       $ 23,170,876.08
5      4        92       $ 22,789,210.42
6      20       95       $ 21,537,795.73
7      4        95       $ 21,054,815.81
8      27       92       $ 20,952,094.28
9      14       95       $ 20,655,911.30
10     2        95       $ 20,533,191.41


# 4. Iterative Forecasting Pipeline (FB Prophet)

In [8]:
# Initialize empty list inside the cell execution state to avoid duplicate caching across notebook re-runs
all_forecasts = []

loop_count = 0
errors_count = 0

for (store_id, dept_id), group_df in df_raw.groupby(["Store", "Dept"]):

    # Process only high-volume combinations identified in the pre-computed set
    if (int(store_id), int(dept_id)) not in top10_set:
        continue

    loop_count += 1
    print(f"\nProcessing [{loop_count:02d}/{TOP_N_COMBINATIONS}] -> Store: {store_id}, Dept: {dept_id} ({len(group_df)} rows)")

    try:
        # Structure dataframe to meet Prophet input schema standards ('ds' and 'y')
        prophet_df = group_df[["Transaction_Date", "Weekly_Sales"]].copy()
        prophet_df = prophet_df.rename(
            columns={"Transaction_Date": "ds", "Weekly_Sales": "y"}
        )
        prophet_df["ds"] = pd.to_datetime(prophet_df["ds"], format="%Y-%m-%d")

        # Enforce chronological ordering and remove sub-weekly granular duplicates
        prophet_df = (
            prophet_df.drop_duplicates(subset=["ds"])
            .sort_values("ds")
            .reset_index(drop=True)
        )

        # Establish the upper chronological boundary of historical training data
        max_train_date_str = prophet_df["ds"].max().strftime("%Y-%m-%d")

        # Map external exogenous regressors if validated in upstream cells
        if available_regressors:
            for reg_col in available_regressors:
                prophet_df[reg_col] = group_df[reg_col].values

        # Initialize Prophet model optimized for weekly retail metrics
        model = Prophet(
            weekly_seasonality=True,
            yearly_seasonality=True,
            daily_seasonality=False,
            seasonality_mode="multiplicative",
            changepoint_prior_scale=0.10,
            seasonality_prior_scale=10.0,
            interval_width=0.95,
        )

        # Condition and attach regressors; suppress standardization for categorical holiday flags
        if available_regressors:
            for reg_col in available_regressors:
                model.add_regressor(reg_col, standardize=(reg_col != "IsHoliday"))

        model.fit(prophet_df)

        # include_history=False restricts prediction output to pure out-of-sample scope
        future = model.make_future_dataframe(
            periods=FORECAST_HORIZON_WEEKS, freq="W-FRI", include_history=False
        )

        # Populate future timeframe using forward-fill for exogenous factors
        if available_regressors:
            for reg_col in available_regressors:
                future[reg_col] = prophet_df[reg_col].iloc[-1]

        forecast = model.predict(future)
        forecast["ds_str"] = forecast["ds"].dt.strftime("%Y-%m-%d")

        # String-based logical filtering to remove any historical date bleedover
        forecast_future_only = forecast[
            forecast["ds_str"] > max_train_date_str
        ].copy()

        if forecast_future_only.empty:
            continue

        # Truncate negative sales values to lower bound zero to maintain financial reality
        for col in ["yhat", "yhat_lower", "yhat_upper"]:
            forecast_future_only[col] = forecast_future_only[col].clip(lower=0.0)

        # Dynamic dimension tagging from loop iteration to preserve relational keys
        forecast_future_only["Store"] = int(store_id)
        forecast_future_only["Dept"] = int(dept_id)

        # Select and map schema structures to match downstream Star Schema requirements
        output_slice = forecast_future_only[[
            "ds_str", "yhat", "yhat_lower", "yhat_upper", "Store", "Dept"
        ]].rename(columns={
            "ds_str": "Transaction_Date",
            "yhat": "Forecasted_Sales",
        })

        output_slice.reset_index(drop=True, inplace=True)
        all_forecasts.append(output_slice)

    except Exception as e:
        errors_count += 1
        print(f"Skipping combination Store={store_id}, Dept={dept_id} due to runtime error: {e}")
        continue

print(f"\nPipeline complete. Processed: {loop_count - errors_count}/{TOP_N_COMBINATIONS} | Errors: {errors_count}")


Processing [01/10] -> Store: 2, Dept: 92 (143 rows)

Processing [02/10] -> Store: 2, Dept: 95 (143 rows)

Processing [03/10] -> Store: 4, Dept: 92 (143 rows)

Processing [04/10] -> Store: 4, Dept: 95 (143 rows)

Processing [05/10] -> Store: 13, Dept: 92 (143 rows)

Processing [06/10] -> Store: 14, Dept: 92 (143 rows)

Processing [07/10] -> Store: 14, Dept: 95 (143 rows)

Processing [08/10] -> Store: 20, Dept: 92 (143 rows)

Processing [09/10] -> Store: 20, Dept: 95 (143 rows)

Processing [10/10] -> Store: 27, Dept: 92 (143 rows)

Pipeline complete. Processed: 10/10 | Errors: 0


# 5. Data Consolidation and Export

In [13]:
if not all_forecasts:
    sys.exit("Error: No valid forecast arrays found in memory. Export aborted.")

# Concatenate isolated loops vertically to maintain structural isolation against cross-joins
df_master = pd.concat(all_forecasts, ignore_index=True)
print(f"After concat: {len(df_master):,} rows")

# Enforce strict analytical data types and standard financial rounding
df_master["Store"] = df_master["Store"].astype(int)
df_master["Dept"] = df_master["Dept"].astype(int)
for col in ["Forecasted_Sales", "yhat_lower", "yhat_upper"]:
    df_master[col] = df_master[col].round(2)

# Deduplicate on composite grain keys to guarantee primary key integrity before load phase
rows_before_dedup = len(df_master)
df_master.drop_duplicates(
    subset=["Transaction_Date", "Store", "Dept"],
    keep="first",
    inplace=True
)
df_master.reset_index(drop=True, inplace=True)
rows_after_dedup = len(df_master)

if rows_before_dedup != rows_after_dedup:
    print(f"Deduplication trace: Removed {rows_before_dedup - rows_after_dedup} duplicate entries.")
else :
  print(f"Dedup check passed — no duplicates found.")

# Chronological sorting to optimize data loading performance into downstream data warehouse models
df_master.sort_values(["Store", "Dept", "Transaction_Date"], inplace=True)
df_master.reset_index(drop=True, inplace=True)

# Map explicit target dimensional schema requirements
df_master = df_master[[
    "Transaction_Date",
    "Store",
    "Dept",
    "Forecasted_Sales",
    "yhat_lower",
    "yhat_upper",
]]

# Force explicit dot decimal layout and omit indexing to mitigate regional engine discrepancies
df_master.to_csv(
    OUTPUT_PATH,
    index=False,
    float_format="%.2f",
    decimal=".",
    encoding="utf-8"
)

print(f"Export execution successful. File saved to location: '{OUTPUT_PATH}'")

After concat: 120 rows
Dedup check passed — no duplicates found.
Export execution successful. File saved to location: 'walmart_sales_predictions_murni_future.csv'


# 6. Master Consolidation and Export

In [15]:
if not all_forecasts:
    sys.exit("Error: No valid forecasts generated.")

# Concat isolated slices vertically to eliminate cross-join data multiplication
df_master = pd.concat(all_forecasts, ignore_index=True)

df_master["Store"] = df_master["Store"].astype(int)
df_master["Dept"] = df_master["Dept"].astype(int)
for col in ["Forecasted_Sales", "yhat_lower", "yhat_upper"]:
    df_master[col] = df_master[col].round(2)

# De-duplication check on composite keys before writing to disk
df_master.drop_duplicates(
    subset=["Transaction_Date", "Store", "Dept"], keep="first", inplace=True
)
df_master.sort_values(["Store", "Dept", "Transaction_Date"], inplace=True)

# Align columns to Power BI schema requirements
df_master = df_master[[
    "Transaction_Date", "Store", "Dept", "Forecasted_Sales", "yhat_lower", "yhat_upper"
]]

# Force universal dot decimals and omit raw indexes to protect regional settings
df_master.to_csv(OUTPUT_PATH, index=False, float_format="%.2f", decimal=".", encoding="utf-8")

# 7. Data Integrity Validation Report

In [16]:
df_verify = pd.read_csv(OUTPUT_PATH)

total_rows = len(df_verify)
min_date_str = df_verify["Transaction_Date"].min()
max_date_str = df_verify["Transaction_Date"].max()
unique_combos = df_verify.groupby(["Store", "Dept"]).ngroups
expected_rows = TOP_N_COMBINATIONS * FORECAST_HORIZON_WEEKS

print(f"Rows Exported : {total_rows} / Expected: {expected_rows}")
print(f"Unique Groups : {unique_combos} / Expected: {TOP_N_COMBINATIONS}")
print(f"Timeline Scope: {min_date_str} to {max_date_str}")

# Audit trail gate verification
historical_leak = df_verify[df_verify["Transaction_Date"] < "2012-11-01"]
duplicate_keys = df_verify.duplicated(subset=["Transaction_Date", "Store", "Dept"]).sum()

if not historical_leak.empty or duplicate_keys > 0:
    print(f"Validation Failed: {len(historical_leak)} historical rows leaked, {duplicate_keys} duplicates found.")
else:
    print("Validation Passed: Output schema is unique and correctly constrained to out-of-sample data.")

Rows Exported : 120 / Expected: 120
Unique Groups : 10 / Expected: 10
Timeline Scope: 2012-11-02 to 2013-01-18
Validation Passed: Output schema is unique and correctly constrained to out-of-sample data.
